# Hands-on Lab: Analyzing Historical Stock/Revenue Data and Building a Dashboard

**Before you upload:** Kernel → Restart & Run All. Wait until every cell has output. Then download this `.ipynb` and submit that file.

The grader checks saved outputs for:
- `tesla_data.head()` with 5 rows
- `tesla_revenue.tail()`
- `gme_data.head()` with 5 rows
- `gme_revenue.tail()`
- Tesla graphs from `make_graph`
- GameStop graphs from `make_graph`

In [1]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## Define Graphing Function

In [2]:
def make_graph(stock_data, revenue_data, stock):
    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        subplot_titles=("Historical Share Price", "Historical Revenue"),
        vertical_spacing=0.3,
    )

    stock = stock_data.copy()
    rev = revenue_data.copy()
    stock["Date"] = pd.to_datetime(stock["Date"], utc=True).dt.tz_localize(None)
    rev["Date"] = pd.to_datetime(rev["Date"], utc=True).dt.tz_localize(None)

    stock_data_specific = stock[stock["Date"] <= "2021-06-14"]
    revenue_data_specific = rev[rev["Date"] <= "2021-04-30"]

    fig.add_trace(
        go.Scatter(
            x=stock_data_specific["Date"],
            y=stock_data_specific["Close"].astype("float"),
            name="Share Price",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=revenue_data_specific["Date"],
            y=revenue_data_specific["Revenue"].astype("float"),
            name="Revenue",
        ),
        row=2,
        col=1,
    )

    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price ($US)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($US Millions)", row=2, col=1)
    fig.update_layout(
        showlegend=False,
        height=900,
        title=stock if isinstance(stock, str) else "Stock",
        xaxis_rangeslider_visible=True,
    )
    fig.show()

In [3]:
# Fix: the previous cell reused the name `stock` for a DataFrame.
# Redefine make_graph so the title argument stays a string.

def make_graph(stock_data, revenue_data, stock_name):
    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        subplot_titles=("Historical Share Price", "Historical Revenue"),
        vertical_spacing=0.3,
    )

    s = stock_data.copy()
    r = revenue_data.copy()
    s["Date"] = pd.to_datetime(s["Date"], utc=True).dt.tz_localize(None)
    r["Date"] = pd.to_datetime(r["Date"], utc=True).dt.tz_localize(None)

    stock_data_specific = s[s["Date"] <= "2021-06-14"]
    revenue_data_specific = r[r["Date"] <= "2021-04-30"]

    fig.add_trace(
        go.Scatter(
            x=stock_data_specific["Date"],
            y=stock_data_specific["Close"].astype("float"),
            name="Share Price",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=revenue_data_specific["Date"],
            y=revenue_data_specific["Revenue"].astype("float"),
            name="Revenue",
        ),
        row=2,
        col=1,
    )

    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price ($US)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($US Millions)", row=2, col=1)
    fig.update_layout(
        showlegend=False,
        height=900,
        title=stock_name,
        xaxis_rangeslider_visible=True,
    )
    fig.show()

## Question 1: Extract Tesla Stock Data

In [4]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")

if tesla_data.empty:
    tesla_data = yf.download("TSLA", period="max", auto_adjust=False, progress=False)
    if isinstance(tesla_data.columns, pd.MultiIndex):
        tesla_data.columns = tesla_data.columns.get_level_values(0)

tesla_data.reset_index(inplace=True)
if "Datetime" in tesla_data.columns and "Date" not in tesla_data.columns:
    tesla_data.rename(columns={"Datetime": "Date"}, inplace=True)

print("tesla_data rows:", len(tesla_data))
tesla_data.head()

tesla_data rows: 4084


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2010-06-29 00:00:00-04:00,1.266667,1.666667,1.169333,1.592667,281494500,0.0,0.0
1,2010-06-30 00:00:00-04:00,1.719333,2.028000,1.553333,1.588667,257806500,0.0,0.0
2,2010-07-01 00:00:00-04:00,1.666667,1.728000,1.351333,1.464000,123282000,0.0,0.0
3,2010-07-02 00:00:00-04:00,1.533333,1.540000,1.247333,1.280000,77097000,0.0,0.0
4,2010-07-06 00:00:00-04:00,1.333333,1.333333,1.055333,1.074000,103003500,0.0,0.0


## Question 2: Extract Tesla Revenue Data

In [5]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
html_data = requests.get(url).text
soup = BeautifulSoup(html_data, "html.parser")

tesla_revenue = pd.DataFrame(columns=["Date", "Revenue"])
for row in soup.find_all("tbody")[1].find_all("tr"):
    col = row.find_all("td")
    if col != []:
        tesla_revenue = pd.concat(
            [tesla_revenue, pd.DataFrame({"Date": [col[0].text], "Revenue": [col[1].text]})],
            ignore_index=True,
        )

tesla_revenue["Revenue"] = tesla_revenue["Revenue"].str.replace(r",|\$", "", regex=True)
tesla_revenue.dropna(inplace=True)
tesla_revenue = tesla_revenue[tesla_revenue["Revenue"] != ""]
tesla_revenue.tail()

,Date,Revenue
48,2010-09-30,31
49,2010-06-30,28
50,2010-03-31,21
52,2009-09-30,46
53,2009-06-30,27


## Question 3: Extract GameStop Stock Data

In [6]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")

if gme_data.empty:
    gme_data = yf.download("GME", period="max", auto_adjust=False, progress=False)
    if isinstance(gme_data.columns, pd.MultiIndex):
        gme_data.columns = gme_data.columns.get_level_values(0)

gme_data.reset_index(inplace=True)
if "Datetime" in gme_data.columns and "Date" not in gme_data.columns:
    gme_data.rename(columns={"Datetime": "Date"}, inplace=True)

print("gme_data rows:", len(gme_data))
gme_data.head()

gme_data rows: 6192


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2002-02-13 00:00:00-05:00,1.620129,1.693350,1.603296,1.691667,76216000,0.0,0.0
1,2002-02-14 00:00:00-05:00,1.712707,1.716074,1.670626,1.683250,11021600,0.0,0.0
2,2002-02-15 00:00:00-05:00,1.683250,1.687458,1.658001,1.674834,8389600,0.0,0.0
3,2002-02-19 00:00:00-05:00,1.666418,1.666418,1.578047,1.607504,7410400,0.0,0.0
4,2002-02-20 00:00:00-05:00,1.615920,1.662209,1.603295,1.662209,6892800,0.0,0.0


## Question 4: Extract GameStop Revenue Data

In [7]:
url_gme = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
html_data_2 = requests.get(url_gme).text
soup_gme = BeautifulSoup(html_data_2, "html.parser")

gme_revenue = pd.DataFrame(columns=["Date", "Revenue"])
for row in soup_gme.find_all("tbody")[1].find_all("tr"):
    col = row.find_all("td")
    if col != []:
        gme_revenue = pd.concat(
            [gme_revenue, pd.DataFrame({"Date": [col[0].text], "Revenue": [col[1].text]})],
            ignore_index=True,
        )

gme_revenue["Revenue"] = gme_revenue["Revenue"].str.replace(r",|\$", "", regex=True)
gme_revenue.dropna(inplace=True)
gme_revenue = gme_revenue[gme_revenue["Revenue"] != ""]
gme_revenue.tail()

,Date,Revenue
57,2006-01-31,1667
58,2005-10-31,534
59,2005-07-31,416
60,2005-04-30,475
61,2005-01-31,709


## Question 5: Plot Tesla Stock Graph

In [8]:
make_graph(tesla_data, tesla_revenue, "Tesla")

## Question 6: Plot GameStop Stock Graph

In [9]:
make_graph(gme_data, gme_revenue, "GameStop")